# Dialogue Dataset Preparation for NPC Training

**Project:** Game-aware NPC: Player Based NPC Behavior Generation for Blockchain Gaming

**Dataset:** Cornell Persuasion For Good (ConvoKit)

**Purpose:** Prepare dialogue training data for fine-tuning to generate Whisper's (NPC merchant) responses.

---

---

# Part 1: Dataset Exploration

The Cornell Persuasion For Good dataset contains dialogues where one participant (persuader) tries to convince another (persuadee) to donate to a charity. This mirrors our NPC merchant scenario where Whisper tries to convince players to purchase items.

## 1.1 Load Dataset

In [23]:
from convokit import Corpus, download

# Download and load the corpus
corpus_path = download("persuasionforgood-corpus")
print(f"Downloaded to: {corpus_path}")

corpus = Corpus(filename=corpus_path)

Dataset already exists at /Users/rameshkrishnan/.convokit/saved-corpora/persuasionforgood-corpus
Downloaded to: /Users/rameshkrishnan/.convokit/saved-corpora/persuasionforgood-corpus


## 1.2 Dataset Overview

In [24]:
# Basic statistics
n_conversations = len(list(corpus.iter_conversations()))
n_utterances = len(list(corpus.iter_utterances()))
n_speakers = len(list(corpus.iter_speakers()))

print("=" * 50)
print("DATASET OVERVIEW")
print("=" * 50)
print(f"Conversations: {n_conversations:,}")
print(f"Utterances:    {n_utterances:,}")
print(f"Speakers:      {n_speakers:,}")
print(f"\nAvg utterances per conversation: {n_utterances / n_conversations:.1f}")

DATASET OVERVIEW
Conversations: 1,017
Utterances:    20,932
Speakers:      1,285

Avg utterances per conversation: 20.6


## 1.3 Conversation Structure

In [25]:
# Examine a sample conversation
sample_conv = list(corpus.iter_conversations())[0]

print("=" * 50)
print("SAMPLE CONVERSATION STRUCTURE")
print("=" * 50)
print(f"Conversation ID: {sample_conv.id}")
print(f"\nConversation metadata keys:")
for key in sample_conv.meta.keys():
    print(f"  - {key}: {sample_conv.meta[key]}")

SAMPLE CONVERSATION STRUCTURE
Conversation ID: 0

Conversation metadata keys:
  - user_er: A3A07QA5U733HQ
  - donation_er: 0.0
  - user_ee: A25L985XCNESXE
  - donation_ee: 0.0
  - intended: 0.2
  - is_annotated: True
  - dialogue_id: 20180904-045349_715_live


In [26]:
# Show conversation flow (first 6 utterances)
print("\n" + "=" * 50)
print("CONVERSATION FLOW (first 6 utterances)")
print("=" * 50)

conv_utts = list(sample_conv.iter_utterances())
for i, utt in enumerate(conv_utts[:6]):
    role = "Persuader" if utt.meta.get('role') == 0 else "Persuadee"
    print(f"\n[{role}]: {utt.text[:120]}{'...' if len(utt.text) > 120 else ''}")


CONVERSATION FLOW (first 6 utterances)

[Persuader]: Good morning. How are you doing today?

[Persuadee]: Hi. I am doing good. How about you?

[Persuader]: I'm doing pretty good for a Tuesday morning. 

[Persuadee]: Haha. Same here, but it really feels like a Monday.

[Persuader]: Ugh yes it does!

[Persuadee]: I can not believe how warm it is already.


## 1.4 Utterance Metadata

In [27]:
# Examine utterance structure
sample_utt = list(corpus.iter_utterances())[0]

print("=" * 50)
print("UTTERANCE METADATA STRUCTURE")
print("=" * 50)
print(f"\nMetadata keys: {list(sample_utt.meta.keys())}")
print(f"\nSample values:")
for key, value in sample_utt.meta.items():
    print(f"  - {key}: {value}")

UTTERANCE METADATA STRUCTURE

Metadata keys: ['user_turn_id', 'role', 'text_by_sent', 'n_sents', 'label_1', 'label_2', 'sentiment']

Sample values:
  - user_turn_id: 0
  - role: 0
  - text_by_sent: Good morning. <s> How are you doing today?
  - n_sents: 2
  - label_1: ['greeting', 'greeting']
  - label_2: [nan, nan]
  - sentiment: {'neg': [0.0, 0.0], 'neu': [0.256, 1.0], 'pos': [0.7440000000000001, 0.0]}


## 1.5 Role Distribution

In this dataset:
- **Role 1**: Persuadee (the person being convinced) → analogous to Player
- **Role 0**: Persuader (the person convincing) → analogous to Whisper (NPC)

In [28]:
# Count utterances by role
role_counts = {0: 0, 1: 0}

for utt in corpus.iter_utterances():
    role = utt.meta.get('role')
    if role in role_counts:
        role_counts[role] += 1

print("=" * 50)
print("ROLE DISTRIBUTION")
print("=" * 50)
print(f"Persuader (Role 0): {role_counts[0]:,} utterances")
print(f"Persuadee (Role 1): {role_counts[1]:,} utterances")
print(f"\nWe need Persuader (Role 0) utterances for training Whisper.")

ROLE DISTRIBUTION
Persuader (Role 0): 10,600 utterances
Persuadee (Role 1): 10,332 utterances

We need Persuader (Role 0) utterances for training Whisper.


## 1.6 Persuasion Strategy Labels

The dataset includes annotated persuasion strategies in `label_1` and `label_2` fields. These strategies inform HOW Whisper should convince players.

In [29]:
# Extract all unique persuasion strategies
strategies = set()

for utt in corpus.iter_utterances():
    label_1 = utt.meta.get('label_1')
    label_2 = utt.meta.get('label_2')
    
    # Labels are stored as lists
    if isinstance(label_1, list):
        for lbl in label_1:
            strategies.add(lbl)
    if isinstance(label_2, list):
        for lbl in label_2:
            strategies.add(lbl)

# Remove NaN if present
strategies = {s for s in strategies if s == s}  # NaN != NaN

print("=" * 50)
print(f"PERSUASION STRATEGIES ({len(strategies)} unique)")
print("=" * 50)
for strategy in sorted(strategies):
    print(f"  - {strategy}")

PERSUASION STRATEGIES (38 unique)
  - acknowledgement
  - agree-donation
  - ask-donate-more
  - ask-donation-amount
  - ask-donation-procedure
  - ask-not-donate-reason
  - ask-org-info
  - ask-persuader-donation-intention
  - closing
  - comment-partner
  - confirm-donation
  - credibility-appeal
  - disagree-donation
  - disagree-donation-more
  - donation-information
  - emotion-appeal
  - foot-in-the-door
  - greeting
  - logical-appeal
  - negative-reaction-to-donation
  - negative-to-inquiry
  - neutral-reaction-to-donation
  - neutral-to-inquiry
  - non-personal-Task-related-inquiry
  - off-task
  - other
  - personal-related-inquiry
  - personal-story
  - positive-reaction-to-donation
  - positive-to-inquiry
  - praise-user
  - proposition-of-donation
  - provide-donation-amount
  - self-modeling
  - source-related-inquiry
  - task-related-inquiry
  - thank
  - you-are-welcome


## 1.7 Labeled vs Unlabeled Utterances

In [30]:
# Count labeled vs unlabeled
labeled_count = 0
unlabeled_count = 0
persuader_labeled = 0
persuader_unlabeled = 0

for utt in corpus.iter_utterances():
    has_label = isinstance(utt.meta.get('label_1'), list)
    is_persuader = utt.meta.get('role') == 0
    
    if has_label:
        labeled_count += 1
        if is_persuader:
            persuader_labeled += 1
    else:
        unlabeled_count += 1
        if is_persuader:
            persuader_unlabeled += 1

print("=" * 50)
print("LABEL DISTRIBUTION")
print("=" * 50)
print(f"Total labeled:       {labeled_count:,}")
print(f"Total unlabeled:     {unlabeled_count:,}")
print(f"")
print(f"Persuader labeled:   {persuader_labeled:,} ← Usable for training")
print(f"Persuader unlabeled: {persuader_unlabeled:,}")

LABEL DISTRIBUTION
Total labeled:       6,136
Total unlabeled:     14,796

Persuader labeled:   3,109 ← Usable for training
Persuader unlabeled: 7,491


## 1.8 Strategy Distribution Among Persuaders

In [31]:
# Count strategies used by persuaders
persuader_strategy_counts = {}

for utt in corpus.iter_utterances():
    # Only persuader utterances with labels
    if utt.meta.get('role') != 0:
        continue
    if not isinstance(utt.meta.get('label_1'), list):
        continue
    
    # Count each strategy
    for label in utt.meta.get('label_1', []):
        persuader_strategy_counts[label] = persuader_strategy_counts.get(label, 0) + 1

print("=" * 50)
print("STRATEGY DISTRIBUTION (Persuader Utterances)")
print("=" * 50)

# Sort by count descending
for strategy, count in sorted(persuader_strategy_counts.items(), key=lambda x: -x[1]):
    pct = count / persuader_labeled * 100
    print(f"  {strategy:40} {count:5} ({pct:5.1f}%)")

STRATEGY DISTRIBUTION (Persuader Utterances)
  credibility-appeal                        1083 ( 34.8%)
  other                                      512 ( 16.5%)
  donation-information                       491 ( 15.8%)
  logical-appeal                             469 ( 15.1%)
  emotion-appeal                             377 ( 12.1%)
  greeting                                   330 ( 10.6%)
  proposition-of-donation                    305 (  9.8%)
  acknowledgement                            298 (  9.6%)
  thank                                      297 (  9.6%)
  task-related-inquiry                       191 (  6.1%)
  source-related-inquiry                     180 (  5.8%)
  praise-user                                174 (  5.6%)
  personal-related-inquiry                   168 (  5.4%)
  self-modeling                              163 (  5.2%)
  foot-in-the-door                           162 (  5.2%)
  personal-story                             153 (  4.9%)
  ask-donation-amount      

## 1.9 Sample Utterances by Strategy

Let's examine what each strategy looks like in practice.

In [32]:
# Show one example per key strategy
key_strategies = [
    'credibility-appeal',
    'emotion-appeal',
    'logical-appeal',
    'praise-user',
    'foot-in-the-door',
    'self-modeling',
    'personal-story'
]

print("=" * 60)
print("SAMPLE UTTERANCES BY STRATEGY")
print("=" * 60)

found_strategies = set()

for utt in corpus.iter_utterances():
    if utt.meta.get('role') != 0:  # Changed from 1 to 0
        continue
    if not isinstance(utt.meta.get('label_1'), list):
        continue
    
    labels = utt.meta.get('label_1', [])
    
    for strategy in key_strategies:
        if strategy in labels and strategy not in found_strategies:
            found_strategies.add(strategy)
            print(f"\n[{strategy}]")
            print(f"\"{utt.text[:200]}{'...' if len(utt.text) > 200 else ''}\"")
    
    if len(found_strategies) >= len(key_strategies):
        break

print(f"\n\nFound {len(found_strategies)}/{len(key_strategies)} strategies")

SAMPLE UTTERANCES BY STRATEGY

[credibility-appeal]
"I like that they're committed to helping children in need. They're very transparent in their work and do great things to help children in underprivileged countries. "

[self-modeling]
"I'm planning on donating most of my earnings today. Would you like to donate as well?"

[logical-appeal]
"Yes it would. Any little bit helps. Thank you for your donation!"

[foot-in-the-door]
"They give you an option at the end of the HIT to donate even 0.01 would be very helpful....a penny couldn't hurt could it?"

[emotion-appeal]
"They work in many countries across the world.  For example, millions of children in Syria grow up facing the daily threat of violence.  A donation could help these children greatly."

[personal-story]
"Oh yeah.  When I first heard about these super well known organizations only actually using about 5% of their proceeds for their services I was shocked.  Save the Children uses a total of 86% of proce..."

[praise-user]
"Tha

## 1.10 Exploration Summary

### Key Findings

| Metric | Value |
|--------|-------|
| Total Conversations | 1,017 |
| Total Utterances | 20,932 |
| Persuader Utterances with Labels | ~3,000 |
| Unique Persuasion Strategies | ~38 |

### Relevant Strategies for NPC Training

| Strategy | NPC Application |
|----------|----------------|
| credibility-appeal | Whisper references his experience, memory, reputation |
| emotion-appeal | Appeal to fear of curses, desire for glory |
| logical-appeal | Cite game mechanics, odds, strategic advantage |
| praise-user | Acknowledge player's wisdom, good decisions |
| foot-in-the-door | Suggest starting with basic hint first |
| self-modeling | Whisper mentions he keeps scrolls himself |
| personal-story | Reference past seekers' fates |


---

# Part 2: Data Filtering

Filter the dataset to extract usable persuader utterances for NPC training.

## 2.1 Define Usable Strategies

Some strategies in the dataset don't translate to NPC dialogue (e.g., 'greeting', 'closing', 'ask-donation-procedure'). We filter to persuasion-relevant strategies.

In [33]:
# Strategies useful for NPC persuasion training
USABLE_STRATEGIES = {
    'credibility-appeal',
    'emotion-appeal',
    'logical-appeal',
    'praise-user',
    'foot-in-the-door',
    'self-modeling',
    'personal-story',
    'proposition-of-donation',  # Direct offers
}

# Strategies to exclude (not useful for persuasion training)
EXCLUDED_STRATEGIES = {
    'greeting',
    'closing',
    'thank',
    'you-are-welcome',
    'acknowledgement',
    'ask-donation-procedure',
    'ask-donation-amount',
    'ask-org-info',
    'donation-information',
    'confirm-donation',
    'other',
}

print(f"Usable strategies: {len(USABLE_STRATEGIES)}")
print(f"Excluded strategies: {len(EXCLUDED_STRATEGIES)}")

Usable strategies: 8
Excluded strategies: 11


## 2.2 Filter Utterances

In [34]:
filtered_data = []

for utt in corpus.iter_utterances():
    # Must be persuader
    if utt.meta.get('role') != 0:
        continue
    
    # Must have labels
    if not isinstance(utt.meta.get('label_1'), list):
        continue
    
    labels = utt.meta.get('label_1', [])
    
    # Filter to usable strategies
    usable_labels = [lbl for lbl in labels if lbl in USABLE_STRATEGIES]
    
    # Must have at least one usable strategy
    if not usable_labels:
        continue
    
    # Must have meaningful text (not too short)
    if len(utt.text.strip()) < 20:
        continue
    
    filtered_data.append({
        'text': utt.text,
        'strategies': usable_labels,
        'sentiment': utt.meta.get('sentiment'),
        'conversation_id': utt.conversation_id
    })

print(f"Filtered utterances: {len(filtered_data)}")

Filtered utterances: 1750


## 2.3 Filtered Data Statistics

In [35]:
# Strategy distribution in filtered data
filtered_strategy_counts = {}

for item in filtered_data:
    for strategy in item['strategies']:
        filtered_strategy_counts[strategy] = filtered_strategy_counts.get(strategy, 0) + 1

print("=" * 50)
print("FILTERED DATA STATISTICS")
print("=" * 50)
print(f"Total usable utterances: {len(filtered_data)}")
print(f"\nStrategy distribution:")

for strategy, count in sorted(filtered_strategy_counts.items(), key=lambda x: -x[1]):
    pct = count / len(filtered_data) * 100
    print(f"  {strategy:30} {count:5} ({pct:5.1f}%)")

FILTERED DATA STATISTICS
Total usable utterances: 1750

Strategy distribution:
  credibility-appeal              1075 ( 61.4%)
  logical-appeal                   469 ( 26.8%)
  emotion-appeal                   377 ( 21.5%)
  proposition-of-donation          304 ( 17.4%)
  praise-user                      169 (  9.7%)
  foot-in-the-door                 162 (  9.3%)
  self-modeling                    157 (  9.0%)
  personal-story                   152 (  8.7%)


## 2.4 Save Filtered Data

In [36]:
import json

# Save filtered data
with open('persuasion_filtered.json', 'w') as f:
    json.dump(filtered_data, f, indent=2)

print(f"✓ Saved {len(filtered_data)} utterances to persuasion_filtered.json")

✓ Saved 1750 utterances to persuasion_filtered.json
